# EnviStor S3 — `sea-level` Bucket Access Template

This notebook shows how to connect to the FIU **EnviStor** Ceph S3 storage and work with the
**`sea-level`** bucket using your own **access key** and **secret key**.

**You will need (ask DoIT / your admin if you don't have these):**

- `ACCESS_KEY` — your S3 access key
- `SECRET_KEY` — your S3 secret key
- Endpoint: `https://envistor.fiu.edu:8580`

> **Important — this is a Ceph RadosGW cluster, not AWS.** It has a few quirks baked into the
> client setup below:
> 1. **Reads** (`GET` / `LIST` / `HEAD`) use **S3v4** signatures + path-style addressing.
> 2. **Writes** (`PUT` / `DELETE` / multipart) require **S3v2 (legacy)** signatures + path-style,
>    otherwise you get `XAmzContentSHA256Mismatch`.
> 3. Listing must use **ListObjectsV1** (`list_objects`), not V2, or you get `SignatureDoesNotMatch`.
>
> The two-client pattern below handles all of this for you.

## 1. Install dependencies

Run once per environment.

In [ ]:
%pip install boto3

## 2. Configure your credentials

Fill in your own keys below. For real deployments prefer environment variables over hard-coding:

```bash
export ENVISTOR_ACCESS_KEY="your-access-key"
export ENVISTOR_SECRET_KEY="your-secret-key"
```

The cell reads from environment variables first and falls back to the inline placeholders.

In [ ]:
import os

# --- EnviStor endpoint (fixed) ---
ENDPOINT = "https://envistor.fiu.edu:8580"
REGION = "us-east-1"

# --- Your credentials (replace, or set env vars) ---
ACCESS_KEY = os.environ.get("ENVISTOR_ACCESS_KEY", "PASTE-YOUR-ACCESS-KEY")
SECRET_KEY = os.environ.get("ENVISTOR_SECRET_KEY", "PASTE-YOUR-SECRET-KEY")

# --- Target bucket ---
BUCKET = "sea-level"

assert ACCESS_KEY and not ACCESS_KEY.startswith("PASTE"), "Set ACCESS_KEY (or ENVISTOR_ACCESS_KEY env var)"
assert SECRET_KEY and not SECRET_KEY.startswith("PASTE"), "Set SECRET_KEY (or ENVISTOR_SECRET_KEY env var)"
print("Config OK — endpoint:", ENDPOINT, "| bucket:", BUCKET)

## 3. Build the S3 clients

Two clients because Ceph RadosGW needs different signature versions for reads vs writes.

In [ ]:
import boto3
from botocore.config import Config as BotoConfig

_base = dict(
    endpoint_url=ENDPOINT,
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    region_name=REGION,
)
_common = dict(retries={"max_attempts": 3, "mode": "adaptive"},
               connect_timeout=10, read_timeout=30)

# READ client: s3v4 + path-style → GET / LIST / HEAD
s3_read = boto3.client(
    "s3", **_base,
    config=BotoConfig(signature_version="s3v4",
                      s3={"addressing_style": "path"}, **_common),
)

# WRITE client: s3v2 (legacy) + path-style → PUT / DELETE / multipart / presigned upload
s3_write = boto3.client(
    "s3", **_base,
    config=BotoConfig(signature_version="s3",
                      s3={"addressing_style": "path"}, **_common),
)

print("Clients ready.")

## 4. Verify access (health check)

Confirm your keys can reach the bucket. `head_bucket` returns nothing on success and raises on failure.

In [ ]:
from botocore.exceptions import ClientError

try:
    s3_read.head_bucket(Bucket=BUCKET)
    print(f"OK — you can access '{BUCKET}'")
except ClientError as e:
    code = e.response.get("Error", {}).get("Code")
    print(f"FAILED to access '{BUCKET}' — error code: {code}")
    print("Check your keys, or ask DoIT whether your token is scoped to this bucket.")
    raise

## 5. List objects

Uses `list_objects` (V1) with pagination via `Marker`. `sea-level` may be empty — that's expected.

In [ ]:
def list_all_objects(bucket, prefix="", page_size=1000):
    """Yield every object dict in a bucket/prefix using ListObjectsV1 pagination."""
    marker = None
    while True:
        params = {"Bucket": bucket, "Prefix": prefix, "MaxKeys": page_size}
        if marker:
            params["Marker"] = marker
        resp = s3_read.list_objects(**params)
        contents = resp.get("Contents", [])
        for obj in contents:
            yield obj
        if resp.get("IsTruncated"):
            marker = resp.get("NextMarker") or (contents[-1]["Key"] if contents else None)
            if not marker:
                break
        else:
            break

objects = list(list_all_objects(BUCKET))
print(f"{len(objects)} objects in '{BUCKET}'")
for o in objects[:20]:
    print(f"  {o['Size']:>12,}  {o['Key']}")
if not objects:
    print("  (bucket is empty — try uploading in the next section)")

## 6. Upload a file (write)

Uses the **write** client (s3v2). Replace the local path with a real file.

In [ ]:
# Option A: upload an in-memory object (good for a quick test)
sample_key = "examples/hello.txt"
s3_write.put_object(Bucket=BUCKET, Key=sample_key, Body=b"hello from EnviStor sea-level\n")
print(f"Uploaded s3://{BUCKET}/{sample_key}")

# Option B: upload a local file (uncomment and set the path)
# local_path = "data/my_dataset.nc"
# dest_key = "tide-gauge/my_dataset.nc"
# s3_write.upload_file(local_path, BUCKET, dest_key)
# print(f"Uploaded {local_path} -> s3://{BUCKET}/{dest_key}")

## 7. Download a file (read)

Uses the **read** client (s3v4).

In [ ]:
# Read straight into memory
obj = s3_read.get_object(Bucket=BUCKET, Key=sample_key)
data = obj["Body"].read()
print("Content:", data.decode("utf-8", errors="replace"))

# Or download to a local file:
# s3_read.download_file(BUCKET, sample_key, "downloaded_hello.txt")
# print("Saved to downloaded_hello.txt")

## 8. Generate a presigned download URL (optional)

Share a temporary link to an object without exposing your keys. Default expiry below is 1 hour.

In [ ]:
url = s3_read.generate_presigned_url(
    "get_object",
    Params={"Bucket": BUCKET, "Key": sample_key},
    ExpiresIn=3600,  # seconds
)
print("Presigned URL (valid 1 hour):")
print(url)

## 9. Cleanup (optional)

Delete the test object you just uploaded. Uses the **write** client.

In [ ]:
s3_write.delete_object(Bucket=BUCKET, Key=sample_key)
print(f"Deleted s3://{BUCKET}/{sample_key}")

## Troubleshooting

| Symptom | Likely cause / fix |
|---------|--------------------|
| `XAmzContentSHA256Mismatch` on upload/delete | You used the **read** (s3v4) client for a write. Use `s3_write`. |
| `SignatureDoesNotMatch` when listing | Don't use `list_objects_v2` on this cluster — use `list_objects` (V1), as in section 5. |
| `403 Forbidden` / `AccessDenied` | Your key isn't scoped to `sea-level`. Ask DoIT to grant access to this bucket. |
| `EndpointConnectionError` | Off the FIU network/VPN, or endpoint/port wrong. Confirm `https://envistor.fiu.edu:8580` is reachable. |
| `NoSuchBucket` | Bucket name typo, or it isn't provisioned for your account. |

**Other EnviStor buckets** (same code, just change `BUCKET`): `remote-sensing`, `legacy-surface-data`, `envistor-osdf`.
Note `legacy-surface-data` is very large (~TB scale) — always use a `prefix` when listing.